# transportive-ai

The driver cells of the post, in its order. Run them from the repo root after the setup in the README. Each section's cells use objects that earlier sections made.

## Store Code and Transaction Histories

In [ ]:
from transportive.data import histories, load

df = load()
H = histories(df)
cancels = int(df["Invoice"].astype(str).str.startswith("C").sum())
print(f"lines {len(df)}, cancellations {cancels}, refunds paired {sum(len(h['refunds']) for h in H)}, invoices {len(H)}")
print(H[0])

## Candidate Workspace

In [ ]:
from transportive.workspace import sandbox

tree = sandbox.up("T01")
print(sandbox.run("T01", tree, "python", "/checks/room.py").stdout.rstrip())
sandbox.down("T01", tree)

## Sealed Ticket Suite

In [ ]:
from transportive.eval.suite import load, seal, valid

T = {t["id"]: t for t in load()}
digest, n = seal()
print(f"tickets {len(T)}, oracles sealed {n}, sha256 {digest[:12]}")
print(f"valid {sum(valid(t) for t in T.values())} of {n}")

## Baseline Failure Study

In [ ]:
import subprocess
from transportive.agents.coder import code

PLAIN, d1 = open("assets/prompts/plain.md").read(), sandbox.up("D1")
meter = {"ticket": "D1", "credits": 0.0, "cap": 6.0, "refusals": 0, "log": []}
code(T["D1"]["text"], PLAIN, "D1", d1, ["search", "read", "edit", "run_tests"], meter, coder="opus")
print(subprocess.run(["git", "-C", str(d1), "diff", "-U1"], capture_output=True, text=True).stdout.rstrip())

In [ ]:
from transportive.eval.suite import oracle

print(oracle(T["D1"], "D1", d1)[1].rstrip())

## Semantic Code Index

In [ ]:
from transportive.index import regions, semantic

print(*[f"tasks.py:{line:<4} {kind:<13} {what}" for line, kind, what in semantic.effects("saleor/saleor/payment/tasks.py")], sep="\n")
print(regions.wrong_context(open("rooms/T24/saleor/payment/utils.py").read(), range(212, 217), {"method": "boleto"}))

## Conflict and Gap Solver

In [ ]:
import yaml
from transportive.engine import context, solver
from transportive.spec import registry
from transportive.spec.canon import canon, fingerprint

spec, draft = (canon(yaml.safe_load(open(f"assets/specs/{n}.yaml"))) for n in ("payment.refund", "payment.refund.draft"))
cores = [tuple(solver.conflict(context.resolve(draft, c)[1])) for c in context.contexts()]
print(*[f"core {list(k)} in {cores.count(k)} contexts" for k in sorted(set(cores)) if k], sep="\n")
go = [c for c in context.contexts() if context.resolve(spec, c)[0] == "go"]
free = ["/".join(c.values()) for c in go if solver.gaps(context.resolve(spec, c)[1], ["same_means"])]
print(f"[NEEDS CLARIFICATION: no rule fixes same_means in {', '.join(free)}]")
left = [c for c in go if solver.conflict(context.resolve(spec, c)[1])]
print(registry.publish(spec, {"payments", "legal", "retail", "warehouse"}, left).as_posix())

## Transport Between Contexts

In [ ]:
from decimal import Decimal
from transportive.engine import transport as law

gbp = Decimal("100.00")
print(law.divide(gbp, 3), law.holonomy(gbp, 3, {**law.LAWS, ("charge", "installments"): law.divide}))
print(law.split(gbp, 3), law.holonomy(gbp, 3), law.transport([Decimal(2), Decimal(3)], ["order_lines", "stock"]))
try:
    law.transport(Decimal("10"), ["card", "boleto"])
except ValueError as e:
    print(e)

## Coordination Kernel

In [ ]:
from transportive.coord import queue
from transportive.coord.leases import Leases
from transportive.verify.rings import verify

a, b = Leases("coder-a"), Leases("coder-b")
print(a.take(["code.refund_within_charged", "saleor.payment.gateway.request_refund_action"]),
      b.take(["code.refund_within_charged"]))
rules = [r for t in ("T29", "T36") for r in context.resolve(spec, T[t]["context"])[1]]
print({t: verify(t, f"rooms/{t}", rules, (1, 2, 3, 4)).ok for t in ("T29", "T36")})
merged, rejected = queue.drain("rooms/main", [(t, queue.git(f"rooms/{t}", "diff").stdout) for t in ("T29", "T36")],
                               lambda repo, rings: verify("main", repo, rules, rings))
print(merged, *rejected, sep="\n")

## Protocol Model Checker

In [ ]:
from transportive.verify import model

for path in ("saleor/saleor/payment/tasks.py", "rooms/T01/saleor/payment/tasks.py"):
    states, trace = model.check(*model.release_task(*model.switches(open(path).read())))
    print(f"{path}: {states} states", *(trace or ["no second refund request"]), sep="\n  ")
for switches in ((True, False), (False, True)):
    print("lock, recheck", switches, "alone:", len(model.check(*model.release_task(*switches))[1]), "steps")

## Mutation Gate

In [ ]:
from pathlib import Path
from transportive.verify import mutate

task, fn = Path("rooms/T01/saleor/payment/tasks.py"), "transaction_release_funds_for_checkout_task"
rules = context.resolve(spec, T["T01"]["context"])[1]
print(len(mutate.mutants(task.read_text(), fn)), "mutants")
print(mutate.kill_matrix(task, fn, lambda: verify("T01", "rooms/T01", rules, (1, 2, 3, 4))))

## Runtime Guard Compiler

In [ ]:
import json
from transportive.guard import monitor, schema

guarded = [r for r in spec.rules if r.db]
print(schema.migration(guarded, "0072_transaction_date_indexes").rstrip())
blocked = monitor.shadow(json.load(open("results/replay/A1.json")), guarded)
print(len(blocked), "refused:", sorted({rule for _, rule in blocked}))

## Traceability Ledger

In [ ]:
from transportive.ledger.trace import Ledger

v11 = registry.amend(spec, yaml.safe_load(open("assets/decisions/D-003.yaml")))
moved = lambda c: solver.changed(context.resolve(spec, c)[1], context.resolve(v11, c)[1]) is not None
print(v11.version, fingerprint(v11), ["/".join(c.values()) for c in context.contexts() if moved(c)])
print(Ledger().impact([t["context"] for t in T.values() if moved(t["context"])]))

## Delivery Loop

In [ ]:
from transportive.agents import team
from transportive.loop.deliver import deliver

meter = {"ticket": "T01", "credits": 0.0, "cap": 6.0, "refusals": 0, "log": []}
build = lambda t, pack, room, tree: team.build(t, pack, room, tree, meter, Leases("architect"))
first, again = deliver(T["T01"], v11, build, Ledger()), deliver(T["T01"], v11, build, Ledger())
print([json.loads(line)["key"] for line in open("results/journal/opus/1.1/T01.jsonl")], first == again)
print(first["status"], first["rounds"], first["patch_sha256"])

In [ ]:
print(queue.git("rooms/T01", "diff", "-U0", "--", "saleor/payment/tasks.py").stdout.rstrip())

## Evaluation Harness

In [ ]:
from transportive.eval import harness

recs = [json.loads(line) for line in open("results/story.jsonl")]
for arm, coder in [(a, c) for a in harness.ARMS for c in ("opus", "qwen")]:
    mine = [r for r in recs if (r["arm"], r["coder"]) == (arm, coder)]
    k = sum(r["hidden_pass"] for r in mine)
    low, high = harness.wilson(k, len(mine))[1:]
    print(arm, coder, f"{k}/{len(mine)} [{low:.2f}, {high:.2f}] gw {harness.green_but_wrong(mine):.3f} double",
          sum(r["violation"] == "duplicate_transfer" for r in mine))
won = {(r["arm"], r["coder"], r["ticket"]): r["hidden_pass"] for r in recs}
ids = sorted({r["ticket"] for r in recs})
print(harness.mcnemar([won["A4", "qwen", t] for t in ids], [won["A1", "opus", t] for t in ids]))